# 8장 — LoRA: 살짝만 조련하기 🪶

웹 챕터의 두 주장을 numpy 로 검증합니다:
1. **rank가 낮으면 얇은 두 행렬의 곱으로 충분하다** — 저장량을 직접 세어 보고
2. **r 는 용량 다이얼이다** — 진짜 변화(ΔW)의 rank가 2일 때, r=1 로는 끝까지 못 배우고
   r=2 면 완벽히 배우는 것을 학습 곡선으로 확인합니다

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch8')

## 1. rank 직관 — 큰 표, 얇은 정보

열 하나(d×1)와 행 하나(1×d)를 곱하면 d×d 짜리 거대한 표가 나옵니다 — 하지만 rank 는 1.
웹 실험 ①의 "모든 행이 기준 행의 배수"가 정확히 이 구조예요.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

d = 100
col = rng.normal(size=(d, 1))   # 열 하나
row = rng.normal(size=(1, d))   # 행 하나
big = col @ row                  # 100×100 = 숫자 10,000개짜리 표!

print(f'표의 크기: {big.shape} — 숫자 {big.size:,}개')
print(f'그런데 rank = {np.linalg.matrix_rank(big)}')
print(f'복원에 필요한 저장량: 열 {col.size} + 행 {row.size} = {col.size + row.size}개 (원본의 {big.size // (col.size + row.size)}분의 1)')

# rank r 짜리는 얇은 두 행렬 (d×r)·(r×d) 로 — LoRA 의 B·A 가 이 모양
r = 3
B, A = rng.normal(size=(d, r)), rng.normal(size=(r, d))
assert np.linalg.matrix_rank(B @ A) == r
print(f'\nB({d}×{r}) @ A({r}×{d}) 의 rank = {np.linalg.matrix_rank(B @ A)} — 얇은 곱으로 원하는 rank를 만든다')

## 2. 미니 LoRA 훈련 — r 는 용량 다이얼

설정: 조련이 요구하는 "진짜 변화" ΔW 가 **rank 2** 라고 합시다(두 방향의 행동 변화 —
예: '답변체로 말하기' + '간결하게').

W 는 얼려 두고, B·A 를 경사하강으로 ΔW 에 맞춰 봅니다 — **r=1 과 r=2 로 각각**.
손실 = (BA − ΔW)² 의 평균, 기울기는 2부의 연쇄법칙으로 손으로 구한 식입니다.

In [ ]:
d = 60
# 진짜 변화 ΔW: rank 2 (방향 두 개의 합)
delta_true = (rng.normal(size=(d, 1)) @ rng.normal(size=(1, d))
              + rng.normal(size=(d, 1)) @ rng.normal(size=(1, d)))
print(f'진짜 변화 ΔW 의 rank = {np.linalg.matrix_rank(delta_true)}')

def train_lora(r, steps=800, lr=0.2, seed=7):
    rng2 = np.random.default_rng(seed)
    B = rng2.normal(size=(d, r)) * 0.1
    A = rng2.normal(size=(r, d)) * 0.1
    curve = []
    for _ in range(steps):
        err = B @ A - delta_true      # 지금의 오차
        curve.append(float((err ** 2).mean()))
        gB = err @ A.T / d            # ∂손실/∂B (연쇄법칙 — 2부. /d 는 눈금 맞추기)
        gA = B.T @ err / d            # ∂손실/∂A
        B -= lr * gB
        A -= lr * gA
    curve.append(float(((B @ A - delta_true) ** 2).mean()))
    return curve

curve_r1 = train_lora(r=1)
curve_r2 = train_lora(r=2)
print(f'최종 손실 — r=1: {curve_r1[-1]:.4f} (바닥에 못 닿음) / r=2: {curve_r2[-1]:.2e} (사실상 0)')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 3.2))
plt.plot(curve_r1, color='#f59e0b', linewidth=1.2, label='r=1 — 용량 부족 (변화는 rank 2인데)')
plt.plot(curve_r2, color='#10b981', linewidth=1.2, label='r=2 — 딱 맞는 용량')
plt.xlabel('훈련 스텝')
plt.ylabel('손실 (BA − ΔW)²')
plt.yscale('log')
plt.title('r 는 용량 다이얼 — 변화의 rank 만큼은 있어야 바닥에 닿는다')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**관찰**: r=1(주황)은 진짜 변화의 방향이 2개인데 1개만 담을 수 있어 어느 수준에서 멈춥니다.
r=2(초록)는 거의 0까지 — **r 이 변화의 rank 이상이면 얇아도 완벽**해요.

## 3. 검증 + 어댑터 영수증

In [ ]:
full = d * d
lora_r2 = 2 * d * 2
print(f'전체 파인튜닝: {full:,}개 vs LoRA(r=2): {lora_r2}개 — ×{full // lora_r2} 절약')

# 검증 1 — r=2 는 거의 완벽하게 배웠다
assert curve_r2[-1] < 1e-3, 'r=2 손실은 거의 0이어야 한다!'
# 검증 2 — r=1 은 용량 부족으로 바닥에 못 닿았다 (r=2보다 훨씬 높은 곳에서 정체)
assert curve_r1[-1] > curve_r2[-1] * 100, 'r=1 은 rank 2짜리 변화를 다 담을 수 없어야 한다!'
# 검증 3 — 저장 절약 배율
assert full // lora_r2 == 15, f'd=60, r=2 → 3600÷240 = 15배 절약'
print('\n✔ 확인: 변화가 저차원이면 얇은 B·A로 충분하고, r는 그 용량의 다이얼이다')

## 4. 관찰 정리 + 🤖 AI 연결

- 열×행 곱 하나가 숫자 1만 개짜리 표를 만들지만 rank는 1 — **크기와 정보량은 다르다**(1부 9장).
- LoRA 훈련 = W 를 얼리고 B·A 만 경사하강(기울기 식은 2부의 연쇄법칙 두 줄).
- **r 는 용량 다이얼**: 변화의 rank 미만이면 못 담고, 이상이면 얇아도 완벽.
  실무의 r 4~16 은 "조련 변화는 대개 그 정도로 얇다"는 경험치입니다.
- 진짜 LoRA와 다른 점: 손실이 (BA−ΔW)² 이 아니라 3장 SFT·7장 DPO 의 손실 그대로이고,
  ΔW 를 미리 아는 게 아니라 그 손실이 이끄는 방향으로 B·A 가 움직입니다 — 원리는 동일.

**다음 챕터**: 9장 — 조련이 잘 됐는지 어떻게 아나? 성적표와 부작용(아첨·망각).

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch8')`.